## Surrogate Attack: 1D-CNN

Train 1D-CNN surrogate → generate whitebox adv → evaluate transfer to tree models (baseline + AT)

### 1. Import

In [1]:
import os, sys, time, math
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import joblib
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from torch.utils.data import DataLoader, TensorDataset

import warnings
warnings.filterwarnings('ignore')

from art.estimators.classification import (
    PyTorchClassifier, SklearnClassifier, CatBoostARTClassifier,
)
from art.estimators.classification import XGBoostClassifier as XGBoostARTClassifier

from art_generator import (
    FGSMAttackGenerator, PGDAttackGenerator, CWAttackGenerator,
    DeepFoolAttackGenerator, MIMAttackGenerator,
)
from utils.masking import get_mutate_indices
from utils.paths import load_attack_config

DEVICE = 'cpu'

### 2. 1D-CNN Surrogate architecture

In [2]:
class _Surrogate1DCNN(nn.Module):
    """1D-CNN for tabular data — reshape features into sequence then apply convolutions.
    Similar sequential processing to LSTM but uses convolution instead of recurrence."""
    def __init__(self, in_dim, n_classes, step_dim=11):
        super().__init__()
        self.step_dim = step_dim
        n_steps = math.ceil(in_dim / step_dim)
        self.pad_len = n_steps * step_dim - in_dim

        self.conv = nn.Sequential(
            # Block 1
            nn.Conv1d(step_dim, 128, kernel_size=3, padding=1),
            nn.BatchNorm1d(128), nn.GELU(), nn.Dropout(0.2),
            # Block 2
            nn.Conv1d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm1d(256), nn.GELU(), nn.Dropout(0.2),
            # Block 3
            nn.Conv1d(256, 128, kernel_size=3, padding=1),
            nn.BatchNorm1d(128), nn.GELU(),
        )
        # Global average pooling → classifier
        self.head = nn.Sequential(
            nn.Linear(128, 64), nn.GELU(), nn.Dropout(0.15),
            nn.Linear(64, n_classes),
        )

    def forward(self, x):
        B, F = x.shape
        if self.pad_len > 0:
            x = nn.functional.pad(x, (0, self.pad_len))
        # (B, features) → (B, step_dim, n_steps) — channels = step_dim
        x = x.view(B, -1, self.step_dim).permute(0, 2, 1)
        x = self.conv(x)           # (B, 128, n_steps)
        x = x.mean(dim=2)          # Global average pooling → (B, 128)
        return self.head(x)


class _Scaled1DCNN(nn.Module):
    """Embed scaler so ART operates in raw feature space."""
    def __init__(self, cnn, mean, scale):
        super().__init__()
        self.cnn = cnn
        self.register_buffer('mean_', torch.tensor(mean, dtype=torch.float32))
        self.register_buffer('scale_', torch.tensor(scale, dtype=torch.float32))
    def forward(self, x):
        return self.cnn((x.float() - self.mean_) / self.scale_)

### 3. Load data

In [3]:
df_test = pd.read_csv('../../datasets/test_shap_66.csv')
X_test = df_test.drop(columns=['Label']).values.astype(np.float32)
y_test = df_test['Label'].values.astype(int)
feature_names = df_test.columns[:-1].tolist()

df_train = pd.read_csv('../../datasets/augment/train_tvae_9600.csv')
X_train = df_train.drop(columns=['Label']).values.astype(np.float32)
y_train = df_train['Label'].values.astype(int)

input_dim = X_train.shape[1]
num_classes = len(np.unique(y_train))

input_metadata = {'feature_names': feature_names, 'label_column': 'Label'}
mutate_indices = get_mutate_indices(df_test)

print(f'Train: {X_train.shape}, Test: {X_test.shape}, Classes: {num_classes}')
print(f'Protected features: {len(mutate_indices)} indices')

Train: (9600, 66), Test: (2028, 66), Classes: 12
Protected features: 12 indices


### 4. Train 1D-CNN surrogate

In [4]:
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train).astype(np.float32)
X_test_sc = scaler.transform(X_test).astype(np.float32)

train_ds = TensorDataset(torch.from_numpy(X_train_sc), torch.from_numpy(y_train).long())
train_loader = DataLoader(train_ds, batch_size=512, shuffle=True)

torch.manual_seed(42)
surrogate = _Surrogate1DCNN(input_dim, num_classes, step_dim=11).to(DEVICE)
optimizer = torch.optim.AdamW(surrogate.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=120)
criterion = nn.CrossEntropyLoss()

for epoch in range(120):
    surrogate.train()
    for xb, yb in train_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(surrogate(xb), yb)
        loss.backward()
        nn.utils.clip_grad_norm_(surrogate.parameters(), 1.0)
        optimizer.step()
    scheduler.step()

    if (epoch + 1) % 20 == 0:
        surrogate.eval()
        with torch.no_grad():
            preds = torch.argmax(surrogate(torch.from_numpy(X_test_sc).to(DEVICE)), dim=1).cpu().numpy()
        acc = accuracy_score(y_test, preds)
        f1 = f1_score(y_test, preds, average='macro')
        print(f'Epoch {epoch+1}: test acc = {acc*100:.2f}%, F1 = {f1:.4f}')

surrogate.eval()
print('\n1D-CNN surrogate training done.')

Epoch 20: test acc = 66.81%, F1 = 0.6467
Epoch 40: test acc = 68.84%, F1 = 0.6696
Epoch 60: test acc = 69.18%, F1 = 0.6767
Epoch 80: test acc = 68.29%, F1 = 0.6662
Epoch 100: test acc = 69.23%, F1 = 0.6766
Epoch 120: test acc = 69.13%, F1 = 0.6754

1D-CNN surrogate training done.


### 5. ART classifier + generate adv on TEST data

In [ ]:
scaled_surrogate = _Scaled1DCNN(surrogate, scaler.mean_, scaler.scale_).to(DEVICE)
scaled_surrogate.eval()

surrogate_clf = PyTorchClassifier(
    model=scaled_surrogate,
    loss=nn.CrossEntropyLoss(),
    input_shape=(input_dim,),
    nb_classes=num_classes,
    clip_values=(float(X_test.min()), float(X_test.max())),
    device_type=DEVICE,
)

preds = np.argmax(surrogate_clf.predict(X_test), axis=1)
print(f'1D-CNN surrogate acc on test: {accuracy_score(y_test, preds)*100:.2f}%')

In [ ]:
SURR_ADV_DIR = '../../adv_samples/adv_eval/surrogate_cnn'
os.makedirs(SURR_ADV_DIR, exist_ok=True)

ATTACK_MAP = {
    'fgsm':     FGSMAttackGenerator,
    'pgd':      PGDAttackGenerator,
    'deepfool': DeepFoolAttackGenerator,
    'cw':       CWAttackGenerator,
    'mim':      MIMAttackGenerator,
}
ATTACK_NAMES = ['fgsm', 'pgd', 'deepfool', 'cw', 'mim']

surrogate_adv = {}
for atk_name in ATTACK_NAMES:
    cfg = load_attack_config(atk_name)
    gen = ATTACK_MAP[atk_name](surrogate_clf, generator_params=cfg)

    print(f'\n=== Generating {atk_name.upper()} ===')
    start = time.time()
    df_adv = gen.generate(X_test, y_test, input_metadata, mutate_indices=mutate_indices)
    print(f'Runtime: {time.time() - start:.2f}s')

    adv = df_adv.drop(columns=['Label']).values.astype(np.float32)
    surrogate_adv[atk_name] = adv

    # Evaluate on surrogate itself
    preds_adv = np.argmax(surrogate_clf.predict(adv), axis=1)
    acc = accuracy_score(y_test, preds_adv) * 100
    print(f'{atk_name.upper()} on surrogate: {acc:.2f}%')

    path = os.path.join(SURR_ADV_DIR, f'cnn_{atk_name}_adv.csv')
    df_adv.to_csv(path, index=False)
    print(f'Saved: {path}')

print('\nDone generating 1D-CNN surrogate adv.')

### 6. Load target tree models (Baseline + AT)

In [ ]:
MODEL_DIR = '../../training/models'
AT_DIR = '../../defense/models'

def wrap_tree_art(model, name):
    clip = (float(np.min(X_test)), float(np.max(X_test)))
    if 'cat' in name.lower():
        return CatBoostARTClassifier(model=model, clip_values=clip, nb_features=X_test.shape[1])
    elif 'xgb' in name.lower():
        return XGBoostARTClassifier(model=model, clip_values=clip, nb_features=X_test.shape[1])
    else:
        return SklearnClassifier(model=model, clip_values=clip)

# Baseline tree models
baseline = {
    'CAT': wrap_tree_art(joblib.load(f'{MODEL_DIR}/framework_cat_TVAE.pkl'), 'cat'),
    'RF':  wrap_tree_art(joblib.load(f'{MODEL_DIR}/framework_rf_TVAE.pkl'), 'rf'),
    'XGB': wrap_tree_art(joblib.load(f'{MODEL_DIR}/framework_xgb_TVAE.pkl'), 'xgb'),
}

# AT tree models (multi-source: ResDNN+LSTM)
at_models = {}
at_cat_path = '../../defense/hybrid/models/framework_cat_multi_at.pkl'
at_rf_path  = f'{AT_DIR}/framework_rf_TVAE_at.pkl'
at_xgb_path = f'{AT_DIR}/framework_xgb_TVAE_at.pkl'

if os.path.exists(at_cat_path):
    at_models['CAT'] = wrap_tree_art(joblib.load(at_cat_path), 'cat')
if os.path.exists(at_rf_path):
    at_models['RF'] = wrap_tree_art(joblib.load(at_rf_path), 'rf')
if os.path.exists(at_xgb_path):
    at_models['XGB'] = wrap_tree_art(joblib.load(at_xgb_path), 'xgb')

# Clean accuracy
print('=== Clean Accuracy ===')
for name in ['CAT', 'RF', 'XGB']:
    acc_b = accuracy_score(y_test, np.argmax(baseline[name].predict(X_test), axis=1)) * 100
    acc_a = accuracy_score(y_test, np.argmax(at_models[name].predict(X_test), axis=1)) * 100 if name in at_models else 0
    print(f'{name:>5s}: Baseline {acc_b:.2f}% | AT {acc_a:.2f}%')

### 7. Evaluate: 1D-CNN surrogate → Tree models (Baseline vs AT)

In [ ]:
print('=== 1D-CNN Surrogate → Tree Models: Baseline vs AT (ResDNN+LSTM) ===')
print(f'{"Attack":>10s} | {"Model":>5s} | {"Base ASR":>10s} | {"AT ASR":>10s} | {"Base Acc":>10s} | {"AT Acc":>10s}')
print('-' * 70)

all_results = []

for name in ['CAT', 'RF', 'XGB']:
    preds_b = np.argmax(baseline[name].predict(X_test), axis=1)
    correct_b = np.where(y_test == preds_b)[0]

    if name in at_models:
        preds_a = np.argmax(at_models[name].predict(X_test), axis=1)
        correct_a = np.where(y_test == preds_a)[0]
    else:
        preds_a, correct_a = None, None

    for atk in ATTACK_NAMES:
        if atk not in surrogate_adv:
            continue
        adv = surrogate_adv[atk]

        # Baseline
        pred_b = np.argmax(baseline[name].predict(adv), axis=1)
        asr_b = np.sum(preds_b[correct_b] != pred_b[correct_b]) / len(correct_b) * 100
        acc_b = accuracy_score(y_test, pred_b) * 100

        # AT
        if preds_a is not None:
            pred_a = np.argmax(at_models[name].predict(adv), axis=1)
            asr_a = np.sum(preds_a[correct_a] != pred_a[correct_a]) / len(correct_a) * 100
            acc_a = accuracy_score(y_test, pred_a) * 100
        else:
            asr_a, acc_a = -1, -1

        print(f'{atk.upper():>10s} | {name:>5s} | {asr_b:>9.2f}% | {asr_a:>9.2f}% | {acc_b:>9.2f}% | {acc_a:>9.2f}%')
        all_results.append({'Model': name, 'Attack': atk.upper(), 'Base ASR': asr_b, 'AT ASR': asr_a, 'Base Acc': acc_b, 'AT Acc': acc_a})

    print('-' * 70)

df_results = pd.DataFrame(all_results)
print('\n=== Average ASR per Model ===')
for name in ['CAT', 'RF', 'XGB']:
    sub = df_results[df_results['Model'] == name]
    print(f'{name:>5s}: Baseline {sub["Base ASR"].mean():.2f}% -> AT {sub["AT ASR"].mean():.2f}%')

### 8. So sánh: MLP surrogate vs 1D-CNN surrogate

In [ ]:
# Load MLP surrogate adv for comparison
MLP_ADV_DIR = '../../adv_samples/adv_eval/surrogate'
mlp_adv = {}
for atk in ATTACK_NAMES:
    path = os.path.join(MLP_ADV_DIR, f'surrogate_{atk}_adv.csv')
    if os.path.exists(path):
        df = pd.read_csv(path)
        mlp_adv[atk] = df.drop(columns=['Label']).values.astype(np.float32)

print('=== CAT: MLP surrogate vs 1D-CNN surrogate (Baseline) ===')
print(f'{"Attack":>10s} | {"MLP ASR":>10s} | {"CNN ASR":>10s} | {"MLP Acc":>10s} | {"CNN Acc":>10s}')
print('-' * 60)

preds_cat = np.argmax(baseline['CAT'].predict(X_test), axis=1)
correct_cat = np.where(y_test == preds_cat)[0]

for atk in ATTACK_NAMES:
    # MLP
    if atk in mlp_adv:
        pred_m = np.argmax(baseline['CAT'].predict(mlp_adv[atk]), axis=1)
        asr_m = np.sum(preds_cat[correct_cat] != pred_m[correct_cat]) / len(correct_cat) * 100
        acc_m = accuracy_score(y_test, pred_m) * 100
    else:
        asr_m, acc_m = 0, 0

    # CNN
    if atk in surrogate_adv:
        pred_c = np.argmax(baseline['CAT'].predict(surrogate_adv[atk]), axis=1)
        asr_c = np.sum(preds_cat[correct_cat] != pred_c[correct_cat]) / len(correct_cat) * 100
        acc_c = accuracy_score(y_test, pred_c) * 100
    else:
        asr_c, acc_c = 0, 0

    print(f'{atk.upper():>10s} | {asr_m:>9.2f}% | {asr_c:>9.2f}% | {acc_m:>9.2f}% | {acc_c:>9.2f}%')

### 9. Confusion matrices: 1D-CNN surrogate → CAT (Baseline vs AT)

In [ ]:
for atk in ATTACK_NAMES:
    if atk not in surrogate_adv:
        continue
    adv = surrogate_adv[atk]

    fig, axes = plt.subplots(1, 2, figsize=(18, 7))
    fig.suptitle(f'CAT vs 1D-CNN Surrogate {atk.upper()}: Baseline vs AT', fontsize=14)

    for ax, (label, clf) in zip(axes, [('Baseline', baseline['CAT']), ('AT (ResDNN+LSTM)', at_models.get('CAT'))]):
        if clf is None:
            ax.set_title('AT model not found')
            continue
        pred = np.argmax(clf.predict(adv), axis=1)
        cm = confusion_matrix(y_test, pred)
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax)
        acc = accuracy_score(y_test, pred) * 100
        ax.set_title(f'{label} (acc={acc:.1f}%)')
        ax.set_xlabel('Predicted')
        ax.set_ylabel('True')
    plt.tight_layout()
    plt.show()